# Level 1: Prompt Engineering (Zero- & Few-Shot Tuning)

Prompt engineering is the art of crafting inputs to guide an LLM to produce the desired output. It's the quickest way to "tune" a model because it doesn't require any changes to the model's weights. It's all about how you ask.

This notebook explores the fundamental techniques of prompt engineering.

### Setup

First, let's install `transformers` and load a pre-trained model. We'll use `Mistral-7B-Instruct-v0.2`, a powerful instruction-following model. Note that running this requires a capable GPU.

In [ ]:
!pip install transformers torch accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load tokenizer and model
# Note: Loading a 7B model requires significant RAM/VRAM.
# If you have limited resources, consider using a smaller model like 'distilgpt2'.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

### 1.1 Prompt Design Principles

Good prompts are clear, specific, and provide context.

#### Bad Prompt (Vague)

In [ ]:
text = "Explain LLM tuning."
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

#### Good Prompt (Specific, with Role Prompting)

In [ ]:
text = "You are an expert AI researcher. Explain the concept of LLM fine-tuning to a software engineer, focusing on the difference between full fine-tuning and PEFT."
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

#### Chain-of-Thought (CoT) Prompting

CoT encourages the model to break down a problem into steps, improving its reasoning.

In [ ]:
cot_prompt = """
Question: A jug has 1000ml of water. I use 250ml for cooking and then 150ml for drinking. I then add 300ml back. How much water is in the jug?

Let's think step by step:
1. Start with the initial amount of water.
2. Subtract the amount used for cooking.
3. Subtract the amount used for drinking.
4. Add the amount that was put back.
5. Calculate the final amount.

Answer:
"""
inputs = tokenizer(cot_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 1.2 Few-Shot Prompting

Provide examples (shots) in the prompt to show the model what you want.

In [ ]:
few_shot_prompt = """
Translate the following English phrases to French:

sea otter -> loutre de mer
peppermint -> menthe poivrée
cheese -> fromage
plush giraffe -> girafe en peluche
"""
inputs = tokenizer(few_shot_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 1.3 Output Formatting

You can instruct the model to return output in a specific format like JSON.

In [ ]:
json_prompt = """
Extract the name, company, and job title from the following text. Return the result as a single JSON object with keys 'name', 'company', and 'title'.

Text: 'Jane Doe recently joined Acme Corporation as the new Chief Technology Officer.'

JSON output:
"""
inputs = tokenizer(json_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))